# Logo Detector - Strict V4.1 Inference Only

This version is based on v4, with targeted fixes:

- OCR is **exact keyword only**. No fuzzy matching, so random text like `CHANGE` should not become `asus`.
- OCR remains strong for text logos like Sony, Lenovo, Huawei, Micromax when the text is actually read.
- Visual logo detection stays strict, but icon-first brands like Apple can pass at lower confidence when the box is small/plausible.
- Video uses temporal confirmation: visual-only labels must persist across multiple nearby frames before they are treated as confirmed. OCR exact hits can confirm immediately.

Expected output key: `strict_fusion_version: v4.1`.

If you see `logo_detections` or `balanced_v5`, you are running an older notebook.

In [ ]:
# CELL 1 | Install dependencies and mount Drive
from google.colab import drive
drive.mount('/content/drive')

!apt-get -qq update
!apt-get -qq install -y tesseract-ocr
!pip install -q "ultralytics>=8.3.0" opencv-python-headless pillow pandas pyyaml pytesseract easyocr

import subprocess, torch
try:
    r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
    print('GPU:', r.stdout.strip() if r.returncode == 0 else 'No GPU visible; CPU inference works but video will be slow.')
except Exception as exc:
    print('GPU check skipped:', exc)
print('STRICT_FUSION_VERSION = v4.1')
print('Torch CUDA:', torch.cuda.is_available())


In [ ]:
# CELL 2 | Load Drive artifacts
from pathlib import Path
import json, yaml

DRIVE_OUT = Path('/content/drive/MyDrive/logo_detection_colab_outputs_new')
if not DRIVE_OUT.exists():
    candidates = list(Path('/content/drive/MyDrive').glob('**/logo_detection_colab_outputs_new'))
    if candidates:
        DRIVE_OUT = candidates[0]

WEIGHTS_PATH = DRIVE_OUT / 'best_logo_detector.pt'
CLASS_MAP_PATH = DRIVE_OUT / 'class_mapping.json'
DATA_YAML_PATH = DRIVE_OUT / 'data.yaml'
OUT_DIR = Path('/content/logo_detection_strict_v4_1_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Artifact folder:', DRIVE_OUT)
print('Weights exists:', WEIGHTS_PATH.exists(), WEIGHTS_PATH)
print('Class map exists:', CLASS_MAP_PATH.exists(), CLASS_MAP_PATH)
print('Data YAML exists:', DATA_YAML_PATH.exists(), DATA_YAML_PATH)
if not WEIGHTS_PATH.exists():
    raise FileNotFoundError('best_logo_detector.pt not found. Add shared folder shortcut to My Drive or edit DRIVE_OUT.')

if CLASS_MAP_PATH.exists():
    info = json.loads(CLASS_MAP_PATH.read_text())
    LOGO_CLASSES = info.get('names') or info.get('kept_classes') or []
elif DATA_YAML_PATH.exists():
    data = yaml.safe_load(DATA_YAML_PATH.read_text()) or {}
    names = data.get('names', [])
    LOGO_CLASSES = list(names.values()) if isinstance(names, dict) else list(names)
else:
    LOGO_CLASSES = []

CLASS_TO_IDX = {name: i for i, name in enumerate(LOGO_CLASSES)}
IDX_TO_CLASS = {i: name for name, i in CLASS_TO_IDX.items()}
print('Visual model classes:', LOGO_CLASSES)
print('OCR can still detect text brands that are absent from visual model classes.')


In [ ]:
# CELL 3 | Strict v4.1 OCR + visual fusion detector
from ultralytics import YOLO
from pathlib import Path
import cv2, time, json, re, shutil
import numpy as np
import pandas as pd
import torch
import pytesseract
import easyocr
from IPython.display import display, Image as IPyImage, HTML
from base64 import b64encode
from collections import deque, Counter

STRICT_FUSION_VERSION = 'v4.1'
print('STRICT_FUSION_VERSION =', STRICT_FUSION_VERSION)

BRAND_KEYWORDS = {
    'motorola': ['motorola', 'moto'], 'samsung': ['samsung'], 'oppo': ['oppo'], 'vivo': ['vivo'],
    'htc': ['htc'], 'sony': ['sony'], 'nokia': ['nokia'], 'honor': ['honor'], 'huawei': ['huawei'],
    'asus': ['asus'], 'lg': ['lg'], 'oneplus': ['oneplus', 'one plus', '1plus'], 'apple': ['apple'],
    'xiaomi': ['xiaomi', 'redmi'], 'mi': ['mi'], 'micromax': ['micromax', 'micro max'],
    'lenovo': ['lenovo'], 'gionee': ['gionee'], 'infocus': ['infocus', 'in focus'],
    'ten_or': ['10.or', '10 or', '10or', 'tenor'], 'lava': ['lava'], 'panasonic': ['panasonic'],
    'intex': ['intex'], 'jio': ['jio'], 'reliance': ['reliance'], 'blackberry': ['blackberry'],
    'google': ['google', 'pixel'],
}

ICON_FIRST_BRANDS = {'apple'}
SHORT_BRANDS = {'lg', 'mi', 'htc', 'jio'}
COCO_DEVICE_OBJECTS = {'cell phone', 'laptop', 'tv', 'monitor', 'remote', 'keyboard', 'book'}

# Conservative defaults; Apple gets a controlled low threshold only for small plausible boxes.
VISUAL_CONF_BY_CLASS = {
    'apple': 0.28,
    'lenovo': 0.62,
    'sony': 0.68,
    'samsung': 0.78,
    'huawei': 0.84,
    'asus': 0.86,
    'motorola': 0.96,
    'micromax': 0.70,
    'nokia': 0.78,
    'panasonic': 0.78,
}
DEFAULT_VISUAL_CONF = 0.76
MIN_BOX_AREA_FRAC = 0.00004
MAX_BOX_AREA_FRAC_GENERAL = 0.12
MAX_BOX_AREA_FRAC_ICON = 0.045
MAX_SIDE_FRAC = 0.55

def normalize_text(text):
    return re.sub(r'[^a-z0-9]+', ' ', str(text).lower()).strip()

def compact(text):
    return normalize_text(text).replace(' ', '')

def exact_brand_matches(text):
    norm = normalize_text(text)
    comp = compact(text)
    if len(comp) < 2:
        return []
    hits = []
    for brand, keys in BRAND_KEYWORDS.items():
        for key in keys:
            k_norm = normalize_text(key)
            k_comp = compact(key)
            if brand in SHORT_BRANDS or len(k_comp) <= 3:
                if norm == k_norm:
                    hits.append((brand, key))
            else:
                # Exact standalone word or exact compact text. No fuzzy partial matching.
                if norm == k_norm or comp == k_comp or re.search(rf'\b{re.escape(k_norm)}\b', norm):
                    hits.append((brand, key))
    seen = {}
    for b,k in hits:
        seen.setdefault(b,k)
    return [{'label':b, 'matched_keyword':k} for b,k in seen.items()]

def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
    inter=max(0,ix2-ix1)*max(0,iy2-iy1)
    aa=max(1,(ax2-ax1)*(ay2-ay1)); bb=max(1,(bx2-bx1)*(by2-by1))
    return inter/(aa+bb-inter+1e-6)

def nms(dets, thr=0.35):
    dets=sorted(dets,key=lambda d:d.get('confidence',0),reverse=True)
    kept=[]
    for d in dets:
        if all(d['label']!=k['label'] or iou(d['bbox'],k['bbox'])<thr for k in kept):
            kept.append(d)
    return kept

def area_frac(box,w,h):
    x1,y1,x2,y2=box
    return max(0,x2-x1)*max(0,y2-y1)/max(1,w*h)

def expand(box,w,h,pad=0.16):
    x1,y1,x2,y2=box; bw=x2-x1; bh=y2-y1
    return [max(0,int(x1-bw*pad)),max(0,int(y1-bh*pad)),min(w,int(x2+bw*pad)),min(h,int(y2+bh*pad))]

def easyocr_bbox(points, offset=(0,0), scale=1.0):
    xs=[p[0] for p in points]; ys=[p[1] for p in points]
    ox,oy=offset
    return [int(min(xs)/scale+ox), int(min(ys)/scale+oy), int(max(xs)/scale+ox), int(max(ys)/scale+oy)]

class StrictFusionDetector:
    def __init__(self, weights, imgsz=768, raw_conf=0.20):
        self.logo_model = YOLO(str(weights))
        self.object_model = YOLO('yolov8n.pt')
        self.imgsz = imgsz
        self.raw_conf = raw_conf
        self.names = self.logo_model.names
        self.reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())
        print('Visual model classes:', self.names)

    def rois(self, frame, include_full=True):
        h,w=frame.shape[:2]
        rois=[]
        if include_full:
            rois.append({'source':'full','box':[0,0,w,h]})
        if max(w,h)>=900:
            tile=int(min(max(512,min(w,h)*0.58),760)); stride=max(360,int(tile*0.78))
            xs=sorted(set(list(range(0,max(1,w-tile+1),stride))+[max(0,w-tile)]))
            ys=sorted(set(list(range(0,max(1,h-tile+1),stride))+[max(0,h-tile)]))
            for y in ys:
                for x in xs:
                    rois.append({'source':'tile','box':[x,y,min(w,x+tile),min(h,y+tile)]})
        try:
            res=self.object_model(frame,imgsz=640,conf=0.30,verbose=False)[0]
            for box in res.boxes:
                name=res.names.get(int(box.cls[0]),'')
                if name not in COCO_DEVICE_OBJECTS:
                    continue
                xyxy=[int(v) for v in box.xyxy[0].cpu().numpy().tolist()]
                rois.append({'source':f'object:{name}','box':expand(xyxy,w,h)})
        except Exception as exc:
            print('Object ROI skipped:', exc)
        unique=[]
        for r in rois:
            if all(iou(r['box'],u['box'])<0.90 for u in unique):
                unique.append(r)
        return unique

    def ocr_on_crop(self, crop, offset=(0,0), source='ocr', upscale=1.0):
        hits=[]
        if crop.size == 0:
            return hits
        proc = cv2.resize(crop, None, fx=upscale, fy=upscale, interpolation=cv2.INTER_CUBIC) if upscale != 1.0 else crop
        rgb = cv2.cvtColor(proc, cv2.COLOR_BGR2RGB)
        try:
            for points, text, conf in self.reader.readtext(rgb, detail=1, paragraph=False, text_threshold=0.55, low_text=0.25, width_ths=0.7):
                matches=exact_brand_matches(text)
                if not matches:
                    continue
                for m in matches:
                    if m['label'] in SHORT_BRANDS and conf < 0.78:
                        continue
                    hits.append({'label':m['label'], 'confidence':float(max(conf,0.98)), 'bbox':easyocr_bbox(points, offset, upscale), 'source':source+'_easyocr_exact', 'text':str(text), 'matched_keyword':m['matched_keyword']})
        except Exception:
            pass
        try:
            gray=cv2.cvtColor(proc, cv2.COLOR_BGR2GRAY)
            gray=cv2.bilateralFilter(gray,7,50,50)
            for psm in [11,6,7]:
                data=pytesseract.image_to_data(gray, output_type=pytesseract.Output.DICT, config=f'--psm {psm}')
                for text, conf, x, y, bw, bh in zip(data.get('text',[]),data.get('conf',[]),data.get('left',[]),data.get('top',[]),data.get('width',[]),data.get('height',[])):
                    try: c=float(conf)
                    except Exception: c=-1
                    if c < 40:
                        continue
                    matches=exact_brand_matches(text)
                    if not matches:
                        continue
                    for m in matches:
                        if m['label'] in SHORT_BRANDS and c < 78:
                            continue
                        ox,oy=offset
                        box=[int(x/upscale+ox), int(y/upscale+oy), int((x+bw)/upscale+ox), int((y+bh)/upscale+oy)]
                        hits.append({'label':m['label'], 'confidence':0.99, 'bbox':box, 'source':source+f'_tesseract{psm}_exact', 'text':str(text), 'matched_keyword':m['matched_keyword']})
        except Exception:
            pass
        return hits

    def ocr_candidates(self, frame):
        h,w=frame.shape[:2]
        hits=[]
        hits += self.ocr_on_crop(frame, (0,0), 'full', upscale=1.0)
        if max(h,w) <= 1600:
            hits += self.ocr_on_crop(frame, (0,0), 'full2x', upscale=2.0)
        for r in self.rois(frame, include_full=False):
            x1,y1,x2,y2=r['box']
            hits += self.ocr_on_crop(frame[y1:y2,x1:x2], (x1,y1), r['source'], upscale=2.0)
        return nms(hits,0.25)

    def visual_candidates(self, frame):
        raw=[]
        for r in self.rois(frame):
            x1,y1,x2,y2=r['box']; crop=frame[y1:y2,x1:x2]
            if crop.size == 0:
                continue
            res=self.logo_model(crop,imgsz=self.imgsz,conf=self.raw_conf,verbose=False)[0]
            for b in res.boxes:
                cid=int(b.cls[0]); label=self.names.get(cid, IDX_TO_CLASS.get(cid, str(cid)))
                conf=float(b.conf[0]); bx1,by1,bx2,by2=[int(v) for v in b.xyxy[0].cpu().numpy().tolist()]
                raw.append({'label':label,'confidence':conf,'bbox':[bx1+x1,by1+y1,bx2+x1,by2+y1],'source':'visual_'+r['source']})
        return nms(raw,0.35)

    def verify_visual(self, raw, frame_shape, ocr):
        h,w=frame_shape[:2]
        ocr_labels={x['label'] for x in ocr}
        accepted=[]; rejected=[]
        for d in raw:
            x1,y1,x2,y2=d['bbox']; bw=x2-x1; bh=y2-y1
            af=area_frac(d['bbox'],w,h); label=d['label']
            threshold=VISUAL_CONF_BY_CLASS.get(label,DEFAULT_VISUAL_CONF)
            max_area=MAX_BOX_AREA_FRAC_ICON if label in ICON_FIRST_BRANDS else MAX_BOX_AREA_FRAC_GENERAL
            reasons=[]
            if d['confidence'] < threshold and label not in ocr_labels:
                reasons.append(f'low_conf<{threshold:.2f}')
            if af < MIN_BOX_AREA_FRAC:
                reasons.append('too_small')
            if af > max_area or bw/w > MAX_SIDE_FRAC or bh/h > MAX_SIDE_FRAC:
                reasons.append('too_large')
            if label == 'motorola' and label not in ocr_labels and d['confidence'] < 0.96:
                reasons.append('motorola_requires_text_or_very_high_conf')
            if reasons:
                rejected.append({**d,'reject_reason':','.join(reasons)})
            else:
                accepted.append({**d,'verification':'visual+ocr' if label in ocr_labels else 'visual_strict_icon_ok' if label in ICON_FIRST_BRANDS else 'visual_strict'})
        return nms(accepted,0.30), rejected

    def fuse(self, ocr, visual):
        fused=[]
        for o in ocr:
            fused.append({**o,'verification':'ocr_exact'})
        for v in visual:
            fused.append(v)
        return nms(fused,0.25)

    def annotate(self, frame, confirmed):
        out=frame.copy()
        for d in confirmed:
            x1,y1,x2,y2=[int(v) for v in d['bbox']]
            is_text='ocr' in d.get('verification','') or 'easyocr' in d.get('source','') or 'tesseract' in d.get('source','')
            color=(0,170,255) if is_text else (0,220,0)
            prefix='TEXT' if is_text else 'LOGO'
            cv2.rectangle(out,(x1,y1),(x2,y2),color,2)
            cv2.putText(out,f"{prefix}:{d['label']} {d['confidence']:.2f}",(x1,max(18,y1-6)),cv2.FONT_HERSHEY_SIMPLEX,0.55,color,2)
        return out

    def predict_visual(self, source):
        frame=cv2.imread(str(source)) if isinstance(source,(str,Path)) else source.copy()
        if frame is None:
            raise ValueError(f'Could not read {source}')
        t0=time.perf_counter()
        ocr=self.ocr_candidates(frame)
        raw=self.visual_candidates(frame)
        visual_ok,rejected=self.verify_visual(raw,frame.shape,ocr)
        confirmed=self.fuse(ocr,visual_ok)
        ann=self.annotate(frame,confirmed)
        return {
            'strict_fusion_version': STRICT_FUSION_VERSION,
            'confirmed_brand_detections': confirmed,
            'ocr_exact_candidates': ocr,
            'visual_candidates': raw,
            'accepted_visual_candidates': visual_ok,
            'rejected_visual_candidates': rejected[:100],
            'competition_found': bool(confirmed),
            '_meta': {'latency_ms': round((time.perf_counter()-t0)*1000,1), 'ocr_count':len(ocr), 'visual_raw_count':len(raw), 'visual_accepted_count':len(visual_ok), 'visual_rejected_count':len(rejected)}
        }, ann

    def run_image(self, image_path):
        payload, ann = self.predict_visual(image_path)
        out_path = OUT_DIR / f'{Path(image_path).stem}_strict_v4_1.jpg'
        cv2.imwrite(str(out_path), ann)
        display(IPyImage(filename=str(out_path)))
        print(json.dumps(payload, indent=2))
        return payload, out_path

    def run_video(self, video_path, target_fps=None, slow_factor=1.5, display_width=720, window=12, min_visual_frames=3):
        cap=cv2.VideoCapture(str(video_path)); src_fps=cap.get(cv2.CAP_PROP_FPS) or 30
        total=int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0); w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)); h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        every=1 if target_fps is None else max(1,round(src_fps/target_fps)); out_fps=max(1,src_fps/max(1.0,slow_factor))
        out_path=OUT_DIR/f'{Path(video_path).stem}_strict_v4_1.mp4'; csv_path=OUT_DIR/f'{Path(video_path).stem}_strict_v4_1.csv'
        writer=cv2.VideoWriter(str(out_path),cv2.VideoWriter_fourcc(*'mp4v'),out_fps,(w,h))
        rows=[]; recent=deque(maxlen=window); last=[]; idx=0
        while cap.isOpened():
            ok,frame=cap.read()
            if not ok: break
            if idx%every==0:
                payload,_=self.predict_visual(frame)
                current=payload['confirmed_brand_detections']
                recent.append(current)
                # OCR exact hits are immediate. Visual-only hits need persistence in recent frames.
                counts=Counter()
                for frame_dets in recent:
                    for d in frame_dets:
                        if not str(d.get('verification','')).startswith('ocr'):
                            counts[d['label']] += 1
                stable=[]
                for d in current:
                    if str(d.get('verification','')).startswith('ocr') or 'ocr_exact' in d.get('verification','') or counts[d['label']] >= min_visual_frames:
                        stable.append(d)
                last=stable
                for d in stable:
                    rows.append({'frame':idx,'time_sec':idx/src_fps,**d})
            ann=self.annotate(frame,last); writer.write(ann); idx+=1
            if idx%50==0: print(f'Processed {idx}/{total or "?"} frames')
        cap.release(); writer.release(); pd.DataFrame(rows).to_csv(csv_path,index=False)
        if rows:
            summary=pd.DataFrame(rows).groupby('label').agg(frames=('frame','nunique'),max_conf=('confidence','max'),mean_conf=('confidence','mean')).reset_index()
            print('Video brand summary:'); display(summary)
        display(HTML(f'<video controls width="{display_width}"><source src="data:video/mp4;base64,{b64encode(open(out_path,"rb").read()).decode()}" type="video/mp4"></video>'))
        return out_path,csv_path

logo_model = StrictFusionDetector(WEIGHTS_PATH, imgsz=768, raw_conf=0.20)
print('Strict Fusion v4.1 ready. Output must include strict_fusion_version = v4.1.')


In [ ]:
# CELL 4 | Upload and test images
from google.colab import files
import shutil

uploaded = files.upload()
for image_name in uploaded.keys():
    payload, out_path = logo_model.run_image(image_name)
    copied = DRIVE_OUT / Path(out_path).name
    shutil.copy2(out_path, copied)
    print('Saved annotated image to:', copied)


In [ ]:
# CELL 5 | Upload and test videos
from google.colab import files
import shutil

uploaded = files.upload()
for video_name in uploaded.keys():
    out_video, csv_path = logo_model.run_video(video_name, target_fps=None, slow_factor=1.5, display_width=720, window=12, min_visual_frames=3)
    shutil.copy2(out_video, DRIVE_OUT / Path(out_video).name)
    shutil.copy2(csv_path, DRIVE_OUT / Path(csv_path).name)
    print('Saved video:', DRIVE_OUT / Path(out_video).name)
    print('Saved CSV:', DRIVE_OUT / Path(csv_path).name)
